<a href="https://colab.research.google.com/github/Willemann0511/MiniProjetoML/blob/main/Mini_projeto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import csv
import urllib.request
import re
from datetime import datetime

print("✓ Ambiente configurado e bibliotecas importadas.")

✓ Ambiente configurado e bibliotecas importadas.


In [2]:
def tratar_categoria(nome_categoria):
    """
    Objetivo: Substituir nulos por 'Sem Categoria', passar para minúsculo,
    remover espaços extras e limpar caracteres indesejados (Regex).
    """
    if not nome_categoria or nome_categoria.strip() == "":
        return "Sem Categoria"

    categoria_tratada = nome_categoria.lower().strip()
    categoria_tratada = re.sub(r'[^\w\s]', '', categoria_tratada)

    return categoria_tratada.strip()

def validar_dimensoes(linha):
    """
    Objetivo: Verificar se as dimensões físicas possuem valores nulos.
    Regra de Corte: Se houver qualquer valor nulo, o registro será descartado (retorna False)
    para evitar erros em relatórios logísticos e de frete.
    """
    colunas_dimensao = [
        'product_weight_g',
        'product_length_cm',
        'product_height_cm',
        'product_width_cm'
    ]
    for coluna in colunas_dimensao:
        valor = linha[coluna].strip()
        if not valor or valor == "":
            return False

    return True

def formatar_data(data_string):
    """
    Objetivo: Converter string de data para o formato 'DD/MM/YYYY'.
    """
    if not data_string or data_string.strip() == "":
        return ""

    objeto_data = datetime.strptime(data_string.strip(), "%Y-%m-%d %H:%M:%S")
    data_formatada = objeto_data.strftime("%d/%m/%Y")

    return data_formatada

print("✓ Funções modulares carregadas com sucesso com escopo definido.")

✓ Funções modulares carregadas com sucesso com escopo definido.


In [3]:
def main():
    """
    Função principal que orquestra o download, processamento em memória e
    geração dos relatórios dos datasets da Olist.
    """
    print("Iniciando o pipeline de higienização de dados...\n")

    # Inicialização das variáveis do Relatório de Status Manual
    total_linhas = 0
    total_cancelados = 0
    total_nulos_corrigidos = 0
    total_descartados = 0
    total_pedidos_vazios = 0
    vazios_mas_nao_cancelados = 0

    # Links diretos do GitHub Raw
    url_csv_products = "https://raw.githubusercontent.com/fiesc-junior-prado/mine_projeto_bloco_1/refs/heads/main/olist_products_dataset.csv"
    url_csv_orders = "https://raw.githubusercontent.com/fiesc-junior-prado/mine_projeto_bloco_1/refs/heads/main/olist_orders_dataset.csv"

    # --- PROCESSAMENTO DO DATASET DE PRODUTOS ---
    try:
        resposta_prod = urllib.request.urlopen(url_csv_products)
        linhas_prod = [l.decode('utf-8') for l in resposta_prod.readlines()]
        leitor_prod = csv.DictReader(linhas_prod)

        with open('produtos_limpos.csv', mode='w', encoding='utf-8', newline='') as destino_prod:
            escritor_prod = csv.DictWriter(destino_prod, fieldnames=leitor_prod.fieldnames)
            escritor_prod.writeheader()

            for linha in leitor_prod:
                total_linhas += 1

                if validar_dimensoes(linha):
                    categoria_original = linha['product_category_name'].strip()
                    linha['product_category_name'] = tratar_categoria(linha['product_category_name'])

                    if categoria_original == "":
                        total_nulos_corrigidos += 1

                    escritor_prod.writerow(linha)
                else:
                    total_descartados += 1

    except Exception as e:
        print(f"Erro no processamento de produtos: {e}")


    # --- PROCESSAMENTO DO DATASET DE PEDIDOS ---
    try:
        resposta_ord = urllib.request.urlopen(url_csv_orders)
        linhas_ord = [l.decode('utf-8') for l in resposta_ord.readlines()]
        leitor_ord = csv.DictReader(linhas_ord)

        with open('pedidos_limpos.csv', mode='w', encoding='utf-8', newline='') as destino_ord:
            escritor_ord = csv.DictWriter(destino_ord, fieldnames=leitor_ord.fieldnames)
            escritor_ord.writeheader()

            for linha in leitor_ord:
                total_linhas += 1

                status_pedido = linha['order_status'].strip()
                data_entrega = inline_data = linha['order_delivered_customer_date'].strip()

                if status_pedido == "canceled":
                    total_cancelados += 1

                linha['order_approved_at'] = formatar_data(linha['order_approved_at'])

                if data_entrega == "":
                    total_pedidos_vazios += 1

                    if status_pedido != "canceled":
                        vazios_mas_nao_cancelados += 1

                escritor_ord.writerow(linha)

    except Exception as e:
        print(f"Erro no processamento de pedidos: {e}")

    # --- ÁREA DE RELATÓRIO E EXIBIÇÃO ---
    print("\n=========================================================")
    print("                  RELATÓRIO DE STATUS                    ")
    print("=========================================================")
    print(f"Total combinado de registros lidos: {total_linhas}")
    print(f"Total de registros nulos corrigidos (Categorias): {total_nulos_corrigidos}")
    print(f"Total de produtos descartados (Dimensões nulas): {total_descartados}")
    print(f"Total de pedidos cancelados identificados: {total_cancelados}")

    print("\n---------------------------------------------------------")
    print("          VALIDAÇÃO DA HIPÓTESE DA DIRETORIA             ")
    print("---------------------------------------------------------")
    print(f"Total de pedidos sem data de entrega: {total_pedidos_vazios}")
    print(f"Pedidos sem data de entrega que NÃO estão cancelados: {vazios_mas_nao_cancelados}")

    if vazios_mas_nao_cancelados == 0 and total_pedidos_vazios > 0:
        print("\n✓ HIPÓTESE COMPROVADA:\n  Todas as datas de entrega vazias são obrigatoriamente de pedidos cancelados.")
    else:
        print("\nX HIPÓTESE REFUTADA:\n  Existem pedidos sem data de entrega que NÃO estão cancelados.")
    print("=========================================================")

# Execução automática controlada
if __name__ == "__main__":
    main()

Iniciando o pipeline de higienização de dados...


                  RELATÓRIO DE STATUS                    
Total combinado de registros lidos: 132392
Total de registros nulos corrigidos (Categorias): 609
Total de produtos descartados (Dimensões nulas): 2
Total de pedidos cancelados identificados: 625

---------------------------------------------------------
          VALIDAÇÃO DA HIPÓTESE DA DIRETORIA             
---------------------------------------------------------
Total de pedidos sem data de entrega: 2965
Pedidos sem data de entrega que NÃO estão cancelados: 2346

X HIPÓTESE REFUTADA:
  Existem pedidos sem data de entrega que NÃO estão cancelados.
